# The ALU — Where Arithmetic and Logic Converge

An Arithmetic Logic Unit selects one operation among add, subtract, AND, OR, XOR under control of an **opcode**, producing a result and **status flags** (zero, carry, overflow, sign). This notebook draws the adder-subtractor, a 1-bit ALU slice with the **opcode-selected path lit**, and the full N-bit unit with live flags.

$$Y = f_{\text{op}}(A, B), \qquad \text{flags} = (Z, C, V, N)$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':9})
ON, OFF='#c0392b','#b0b0b0'
def wcol(b): return ON if b else OFF
def wlw(b):  return 2.4 if b else 1.1
def gate_box(ax,x,y,label,w=0.9,h=0.66):
    ax.add_patch(Rectangle((x,y-h/2),w,h,fc='#eef2f7',ec='#34495e',lw=1.3,zorder=2))
    ax.text(x+w/2,y,label,ha='center',va='center',fontsize=7.5,weight='bold',zorder=3)
    return (x,y+h*0.28),(x,y-h*0.28),(x+w,y)
def wire(ax,p0,p1,bit,elbow=True):
    c,lw=wcol(bit),wlw(bit); (x0,y0),(x1,y1)=p0,p1
    if elbow and abs(y0-y1)>1e-6:
        xm=(x0+x1)/2
        ax.plot([x0,xm],[y0,y0],color=c,lw=lw,zorder=1); ax.plot([xm,xm],[y0,y1],color=c,lw=lw,zorder=1); ax.plot([xm,x1],[y1,y1],color=c,lw=lw,zorder=1)
    else: ax.plot([x0,x1],[y0,y1],color=c,lw=lw,zorder=1)
print('primitives ready')


primitives ready


## Adder-Subtractor — One Control Bit Flips the Operation

XOR-ing each $B_i$ with a control line $sub$ and feeding $sub$ as the initial carry computes $A+B$ when $sub=0$ and $A-B$ (via two's complement) when $sub=1$. The single control bit reuses the same adder for both. The schematic lights the XOR inversion path when subtracting.

$$A - B = A + \overline{B} + 1$$


In [2]:
def adder_subtractor(a,b,sub,nbits):
    mask=(1<<nbits)-1; a&=mask; b&=mask
    b_eff = (~b & mask) if sub else b
    raw = a + b_eff + (1 if sub else 0)
    res = raw & mask
    cout = (raw >> nbits) & 1
    fig,ax=plt.subplots(figsize=(8,3.4)); ax.set_xlim(0,10); ax.set_ylim(0,4); ax.axis('off')
    # B xor sub stage
    for i in range(nbits):
        x=1.0+i*1.6
        bi=(b>>i)&1
        _,_,xo=gate_box(ax,x,3.0,'XOR',w=0.7,h=0.5)
        ax.text(x-0.3,3.3,f'B{i}={bi}',fontsize=7,color=wcol(bi),ha='center')
        bxs=(b_eff>>i)&1
        wire(ax,(x+0.7,3.0),(x+0.7,2.3),bxs,elbow=False)
    # sub control rail
    ax.plot([0.5,1.0+nbits*1.6],[3.7,3.7],color=wcol(sub),lw=wlw(sub))
    ax.text(0.2,3.7,f'sub={sub}',fontsize=8,color=wcol(sub),weight='bold',ha='right')
    # adder block
    ax.add_patch(Rectangle((0.8,1.2),nbits*1.6,1.0,fc='#eef2f7',ec='#34495e',lw=1.6))
    ax.text(0.8+nbits*0.8,1.7,f'{nbits}-bit ADDER  (Cin=sub={sub})',ha='center',va='center',fontsize=8.5,weight='bold')
    for i in range(nbits):
        x=1.0+i*1.6; ri=(res>>i)&1
        wire(ax,(x+0.4,1.2),(x+0.4,0.6),ri,elbow=False)
        ax.text(x+0.4,0.35,f'{ri}',fontsize=9,color=wcol(ri),ha='center',weight='bold')
    op = 'A - B' if sub else 'A + B'
    signed = res-(1<<nbits) if (sub and (res>>(nbits-1))&1) else res
    ax.set_title(f'{op}:  {a} {"-" if sub else "+"} {b} = {res}  (cout={cout})',fontsize=10)
    plt.tight_layout(); plt.show()
w_a=widgets.IntSlider(value=5,min=0,max=15,description='A:')
w_b=widgets.IntSlider(value=3,min=0,max=15,description='B:')
w_sub=widgets.ToggleButtons(options=[0,1],value=1,description='sub:')
display(widgets.VBox([w_a,w_b,w_sub]),
        widgets.interactive_output(adder_subtractor,{'a':w_a,'b':w_b,'sub':w_sub,'nbits':widgets.fixed(4)}))


Output()

## 1-Bit ALU Slice — The Opcode Selects the Path

A slice computes all candidate results (sum, AND, OR, XOR) in parallel and a MUX picks one by opcode. The schematic computes every function but **lights only the path the opcode selects**, exactly how a real ALU wastes gates for speed.


In [3]:
OPS={0:'ADD',1:'AND',2:'OR',3:'XOR'}
def alu_slice(A,B,Cin,op):
    s = A^B^Cin; cout=(A&B)|(Cin&(A^B))
    cand={0:s,1:A&B,2:A|B,3:A^B}
    Y=cand[op]
    fig,ax=plt.subplots(figsize=(8,4.2)); ax.set_xlim(0,9); ax.set_ylim(0,5); ax.axis('off')
    ax.text(0.3,4.6,f'A={A}  B={B}  Cin={Cin}',fontsize=9,weight='bold')
    labels=['SUM','AND','OR','XOR']; vals=[s,A&B,A|B,A^B]; ys=[4.0,3.0,2.0,1.0]
    for i,(lab,v,y) in enumerate(zip(labels,vals,ys)):
        sel=(i==op)
        gate_box(ax,1.2,y,lab,w=1.0,h=0.6)
        col=wcol(v) if sel else '#dddddd'; lw=wlw(v) if sel else 1.0
        ax.text(0.9,y,f'{v}',fontsize=8,color=wcol(v) if sel else '#bbb',ha='right',weight='bold')
        ax.plot([2.2,4.5],[y,y],color=col,lw=lw,zorder=1)
        if sel: ax.plot([4.5,5.2],[y,2.5],color=wcol(v),lw=wlw(v),zorder=3)
    body=Polygon([(4.5,1.0),(5.2,1.6),(5.2,3.4),(4.5,4.0)],closed=True,fc='#eef2f7',ec='#34495e',lw=1.5,zorder=2)
    ax.add_patch(body); ax.text(4.85,2.5,'MUX',ha='center',va='center',fontsize=7.5,weight='bold',rotation=90,zorder=3)
    ax.plot([5.2,7.0],[2.5,2.5],color=wcol(Y),lw=wlw(Y),zorder=3)
    ax.scatter([7.0],[2.5],s=44,color=wcol(Y),zorder=4)
    ax.text(7.2,2.5,f'Y={Y}',fontsize=10,color=wcol(Y),weight='bold',va='center')
    ax.text(4.85,0.6,f'op={op} ({OPS[op]})',ha='center',fontsize=9,color='#8e44ad',weight='bold')
    ax.set_title(f'1-bit ALU slice  --  {OPS[op]} selected, Y={Y}',fontsize=10)
    plt.tight_layout(); plt.show()
w_A=widgets.ToggleButtons(options=[0,1],value=1,description='A:')
w_B=widgets.ToggleButtons(options=[0,1],value=1,description='B:')
w_C=widgets.ToggleButtons(options=[0,1],value=0,description='Cin:')
w_op=widgets.Dropdown(options=[(v,k) for k,v in OPS.items()],value=0,description='opcode:')
display(widgets.VBox([w_A,w_B,w_C,w_op]),
        widgets.interactive_output(alu_slice,{'A':w_A,'B':w_B,'Cin':w_C,'op':w_op}))


Output()

## N-Bit ALU With Status Flags

Stacking slices and decoding the result yields the four classic flags: **Z**ero (result is 0), **C**arry (out of MSB), o**V**erflow (signed range exceeded), and **N**egative (MSB set). These flags drive conditional branches in a CPU. Pick operands and opcode to see the result and flags update.

$$Z=\overline{\bigvee Y_i},\quad N=Y_{n-1},\quad V=C_{n}\oplus C_{n-1}$$


In [4]:
def alu_nbit(A,B,op,nbits):
    mask=(1<<nbits)-1; A&=mask; B&=mask
    if op==0:   raw=A+B; Y=raw&mask; C=(raw>>nbits)&1
    elif op==4: raw=A+(~B&mask)+1; Y=raw&mask; C=(raw>>nbits)&1   # SUB
    else:
        Y={1:A&B,2:A|B,3:A^B}[op]; C=0
    Z=int(Y==0); N=(Y>>(nbits-1))&1
    # overflow only meaningful for add/sub
    if op in (0,4):
        sa=(A>>(nbits-1))&1; sb=(B>>(nbits-1))&1 if op==0 else (((~B&mask)>>(nbits-1))&1)
        V=int(sa==sb and ((Y>>(nbits-1))&1)!=sa)
    else: V=0
    names={0:'ADD',4:'SUB',1:'AND',2:'OR',3:'XOR'}
    fig,ax=plt.subplots(figsize=(8,3.2)); ax.set_xlim(0,10); ax.set_ylim(0,4); ax.axis('off')
    ax.add_patch(Rectangle((3,1.2),3,1.8,fc='#eef2f7',ec='#34495e',lw=1.8))
    ax.text(4.5,2.1,f'ALU\n{names[op]}',ha='center',va='center',fontsize=11,weight='bold')
    ax.text(2.7,2.6,f'A={A}\n{format(A,f"0{nbits}b")}',ha='right',fontsize=9,color='#2471a3')
    ax.text(2.7,1.5,f'B={B}\n{format(B,f"0{nbits}b")}',ha='right',fontsize=9,color='#2471a3')
    ax.annotate('',xy=(3,2.5),xytext=(2.0,2.6),arrowprops=dict(arrowstyle='->',lw=1.4,color='#2471a3'))
    ax.annotate('',xy=(3,1.6),xytext=(2.0,1.5),arrowprops=dict(arrowstyle='->',lw=1.4,color='#2471a3'))
    ax.annotate('',xy=(7.0,2.1),xytext=(6,2.1),arrowprops=dict(arrowstyle='->',lw=1.6,color='#c0392b'))
    ax.text(7.1,2.1,f'Y={Y}\n{format(Y,f"0{nbits}b")}',fontsize=10,color='#c0392b',weight='bold',va='center')
    flags=[('Z',Z),('C',C),('V',V),('N',N)]
    for i,(fn,fv) in enumerate(flags):
        x=3.2+i*0.75
        ax.add_patch(Rectangle((x,0.3),0.55,0.55,fc='#c0392b' if fv else '#eee',ec='#333',lw=1.2))
        ax.text(x+0.27,0.57,fn,ha='center',va='center',fontsize=9,color='white' if fv else '#777',weight='bold')
    ax.text(3.2,0.0,'status flags',fontsize=8,color='#555')
    ax.set_title(f'{names[op]}: result {Y}  flags Z={Z} C={C} V={V} N={N}',fontsize=10)
    plt.tight_layout(); plt.show()
w_aa=widgets.IntSlider(value=6,min=0,max=15,description='A:')
w_bb=widgets.IntSlider(value=10,min=0,max=15,description='B:')
w_oo=widgets.Dropdown(options=[('ADD',0),('SUB',4),('AND',1),('OR',2),('XOR',3)],value=0,description='opcode:')
display(widgets.VBox([w_aa,w_bb,w_oo]),
        widgets.interactive_output(alu_nbit,{'A':w_aa,'B':w_bb,'op':w_oo,'nbits':widgets.fixed(4)}))


Output()

## Verifying the Overflow Flag

Signed overflow occurs exactly when two operands of the same sign produce a result of the opposite sign. The cell exhaustively checks 4-bit signed addition and confirms the V flag matches the true signed-range violation for every input pair.


In [5]:
def check_overflow(nbits):
    mask=(1<<nbits)-1; lo=-(1<<(nbits-1)); hi=(1<<(nbits-1))-1
    mism=0; total=0
    for a in range(1<<nbits):
        for b in range(1<<nbits):
            total+=1
            raw=a+b; Y=raw&mask
            sa=(a>>(nbits-1))&1; sb=(b>>(nbits-1))&1; sy=(Y>>(nbits-1))&1
            V=int(sa==sb and sy!=sa)
            asig=a-(1<<nbits) if sa else a; bsig=b-(1<<nbits) if sb else b
            true_ov=int(not (lo <= asig+bsig <= hi))
            if V!=true_ov: mism+=1
    print(f'{nbits}-bit signed add: checked {total} pairs, V-flag mismatches = {mism}')
    print('overflow flag is correct for every input pair' if mism==0 else 'FLAG BUG')
check_overflow(4)


4-bit signed add: checked 256 pairs, V-flag mismatches = 0
overflow flag is correct for every input pair
